# Guruswami--Sudan at the bound: theory versus what actually decodes

§IV-B recovers a *set* of Reed--Solomon codewords from one superposition by list decoding
the per-position histogram. Theory says that works up to

$$u_{\max} \;=\; \left\lfloor \frac{N-1}{K-1} \right\rfloor$$

bundled codewords. This notebook computes that bound for a $(p, m, K)$ you set at the
top, then **measures where decoding actually stops**, and gives you the one knob --- the
interpolation multiplicity $d$ --- that can move the measured reach independently of the
bound.

That the two could differ at all is the point. Every derivation of $u_{\max}$ in §1 is a
**sufficient** condition: each says *if* $t \le u_{\max}$ *then* decoding works. None
proves the converse, so nothing in the theory forbids some $t > u_{\max}$ from decoding
anyway. §3 is where you find out whether any does.

| Section | |
|---|---|
| **1. Theory** | three independent arguments, all landing on the same $u_{\max}$ |
| **2. Actual, $d = 1$** | sweep $t$ until recovery stops being exact |
| **3. Past the bound?** | the same bundles at higher $d$, measured |

Everything calls [streamlit_app/core.py](streamlit_app/core.py), the code the app and
[histogram_recovery.ipynb](histogram_recovery.ipynb) already run, so nothing here can
drift from them.

In [ ]:
import json, math, sys, time
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

REPO = Path.cwd()
sys.path.insert(0, str(REPO / "streamlit_app"))
import core

%matplotlib inline

# ---- set the code here ---------------------------------------------------------
P, M, K = 3, 3, 3            # the field is F_{p^m}; K is the Reed-Solomon dimension
D_LIST  = [1, 2, 3, 4, 5]    # interpolation multiplicities to measure
# --------------------------------------------------------------------------------

ctx = core.build_context(P, M, K)
N, DIM = ctx["N"], ctx["dim"]
u_max = ctx["u_max"]
CACHE = REPO / "streamlit_app" / "data" / f"gs_bound_{P}_{M}_{K}.jsonl"
CACHE.parent.mkdir(parents=True, exist_ok=True)

print(f"field        F_{P}^{M}  ->  N = {N} evaluation points")
print(f"outer code   [n={N}, k={K}] Reed-Solomon over F_{N},  {N**K} messages")
print(f"hypervectors dimension {DIM} = p^(2m)")
print(f"u_max        {u_max}")

## 1. Theory

Three different arguments bound how many codewords one superposition can be unbundled
into. They are not the same argument, which is why $u_{\max}$ feels like a hard wall ---
it is the answer you get three ways.

**(a) Score separation (Lemma 2).** Stage B keeps a candidate $f$ only if it scores the
maximum $dN$: that is, only if the symbol $f(x_j)$ was actually observed at *every*
position $j$. A genuine bundled codeword scores $dN$ by construction. An impostor $g$ can
only score where it agrees with one of the $t$ genuine codewords, and two distinct
polynomials of degree $< K$ agree in at most $K-1$ places, so

$$\mathrm{score}(g) \;\le\; t(K-1).$$

The filter is provably sound while $t(K-1) < N$.

**(b) The witness curve.** Interpolation seeks $Q$ of $(1, K-1)$-weighted degree at most
$dN$ vanishing to order $d$ at every observed point. One such $Q$ always exists in
principle, built from the answer itself:

$$Q^{*}(x, y) \;=\; \prod_{i=1}^{t}\bigl(y - f_i(x)\bigr)^{d},$$

of weighted degree $t\,d\,(K-1)$. It fits the budget when $td(K-1) \le dN$, i.e. when
$t(K-1) \le N$ --- **$d$ cancels**. This is the argument that makes raising $d$ look
pointless. Keep it in mind for §3: it proves $Q^{*}$ stops fitting, not that $Q^{*}$ is
the only curve that would do.

**(c) The stated bound.** $u_{\max} = \lfloor (N-1)/(K-1) \rfloor$, the largest $t$
satisfying (a).

In [ ]:
b = core.bounds(ctx)

print("classical Reed-Solomon radii (properties of the outer code alone)")
print(f"  d_min                        {b['d_min']}")
print(f"  unique-decoding radius       {b['unique_radius']}")
print(f"  Johnson radius               {b['johnson']:.3f}")
print()
print("histogram-recovery bound")
print(f"  u_max = floor((N-1)/(K-1))   {u_max}")
print()
print(f"  {'t':>3} {'t(K-1)':>7} {'vs N':>5}   (a) filter sound?   (b) witness fits?")
for t in range(1, u_max + 4):
    sep  = "yes" if t * (K - 1) < N else "NO"
    fits = "yes" if t * (K - 1) <= N else "NO"
    mark = "   <- u_max" if t == u_max else ""
    print(f"  {t:>3} {t*(K-1):>7} {N:>5}   {sep:>16}   {fits:>16}{mark}")

## 2. What actually decodes, at $d = 1$

Bundle $t$ codewords, each bundle containing the one before, and decode. `exact` means
the recovered set is *precisely* the set that went in --- nothing missing, nothing
spurious.

`n_P` is the number of observed (position, symbol) pairs, which is what interpolation
actually sees. It is at most $tN$, and falls short whenever two bundled codewords happen
to agree somewhere. It is the real constraint count, so §3 uses it rather than $tN$.

In [ ]:
msgs = core.message_list(ctx, u_max + 4)

cum, Ys, n_Ps = [], [], []
acc = np.zeros(DIM, dtype=complex)
for t in range(1, u_max + 4):
    acc = acc + core.encode(ctx, msgs[t - 1])
    Y = core.support_matrix(core.histograms(ctx, acc))
    cum.append(acc.copy())
    Ys.append(Y)
    n_Ps.append(int(Y.sum()))

print(f"{'t':>3} {'n_P':>5} {'tN':>5} {'ell':>4} {'found':>6} {'exact':>6} {'ms':>7}")
for t in range(1, u_max + 3):
    r = core.decode(ctx, cum[t - 1], d=1, ell_min=t, ell_max=t + 2)
    exact = r["messages"] == sorted(msgs[:t])
    mark = "   <- u_max" if t == u_max else ""
    print(f"{t:>3} {n_Ps[t-1]:>5} {t*N:>5} {r['ell']:>4} {len(r['messages']):>6} "
          f"{str(exact):>6} {r['ms']:>7.0f}{mark}")

## 3. Past the bound?

Argument (b) says $d$ cancels. It is sound, but it only exhibits *one* curve. A $Q$ that
factors into the right roots does not have to be $Q^{*}$ --- and whether some *other*
curve exists is a counting question, where $d$ emphatically does not cancel:

$$\underbrace{\#\{(i,j) : i + j(K-1) \le dN,\; j \le \ell\}}_{\text{unknowns, grows like } d^2}
\quad\text{versus}\quad
\underbrace{n_P \cdot \tfrac{d(d+1)}{2}}_{\text{constraints, grows like } d^2}$$

Both sides are quadratic in $d$, so which one wins is a question about constants, not
asymptotics, and it has to be computed rather than argued.

### First: $\ell$ has to be big

The unknown count rises with $\ell$ and then **saturates**, because $j \le dN/(K-1)$ is
forced by the budget regardless of $\ell$. So the only $\ell$ worth testing is the
saturating one:

$$\ell_{\mathrm{sat}}(d) \;=\; \left\lfloor \frac{dN}{K-1} \right\rfloor$$

This matters practically. At $d = 1$ the first $\ell$ admitting a curve is $\ell = t$,
which is why `core.unbundle` walks $\ell$ upward from 2 and stops at the first hit. At
$d = 4$, $\ell_{\mathrm{sat}}$ is far larger --- and larger than `unbundle`'s default
`ell_max = 40`. **The shipped decoder cannot reach these curves at higher $d$ even when
they exist**, so this section calls `core.attempt`, which takes an explicit
$(\text{budget}, \ell)$, instead.

In [ ]:
from sage.coding.guruswami_sudan.interpolation import _monomial_list


def ell_sat(d):
    """Beyond this ell the monomial count stops growing, so no larger ell can help."""
    return d * N // (K - 1)


def counting(t, d):
    """Unknowns and constraints for the interpolation step, at the budget core.py uses."""
    unknowns = len(list(_monomial_list(d * N, ell_sat(d), K - 1)))
    constraints = n_Ps[t - 1] * d * (d + 1) // 2
    return unknowns, constraints


print(f"{'t':>3} {'d':>3} {'ell_sat':>8} {'unknowns':>9} {'constraints':>12} "
      f"{'slack':>7}  curve must exist?")
for t in (u_max, u_max + 1, u_max + 2):
    for d in D_LIST:
        un, con = counting(t, d)
        print(f"{t:>3} {d:>3} {ell_sat(d):>8} {un:>9} {con:>12} {un-con:>7}  "
              f"{'yes' if un > con else 'not guaranteed'}")
    print()

A negative slack does **not** prove no curve exists --- the constraints are rows of a
matrix and need not be independent, so a kernel can appear with the count against it.
That is exactly why the next cell measures rather than concludes.

### The measurement

Each $(d, t)$ below is one real interpolate-and-factor at $\ell_{\mathrm{sat}}(d)$, with
the score filter applied, so anything reported as recovered is certified: it scored the
maximum $dN$ against the observed histogram.

Results are cached to `streamlit_app/data/gs_bound_<p>_<m>_<K>.jsonl`, so re-running is
free and only new combinations cost anything.

**This is the expensive cell.** Cost climbs steeply --- the interpolation matrix is
roughly $n_P\frac{d(d+1)}{2}$ by $\ell_{\mathrm{sat}}$-many columns, so it grows like
$d^4$. Trim `D_LIST` or `T_WINDOW` if you are impatient.

In [ ]:
T_WINDOW = [u_max, u_max + 1, u_max + 2]       # the interesting neighbourhood


def cached():
    if not CACHE.exists():
        return {}
    rows = [json.loads(l) for l in CACHE.read_text().splitlines() if l.strip()]
    return {(r["d"], r["t"]): r for r in rows}


def measure(d, t, store):
    """One interpolate-factor-filter at multiplicity d on the bundle of t codewords."""
    if (d, t) in store:
        return store[(d, t)]
    un, con = counting(t, d)
    t0 = time.time()
    a = core.attempt(ctx, Ys[t - 1], budget=d * N, ell=ell_sat(d), d=d)
    row = {"p": P, "m": M, "K": K, "d": d, "t": t, "ell": ell_sat(d),
           "n_P": n_Ps[t - 1], "unknowns": un, "constraints": con,
           "kernel": int(a["kernel"]), "roots": int(a["roots"]),
           "found": len(a["messages"]), "exact": a["messages"] == sorted(msgs[:t]),
           "sec": round(time.time() - t0, 1)}
    with CACHE.open("a") as fh:
        fh.write(json.dumps(row) + "\n")
    store[(d, t)] = row
    return row


store = cached()
grid, crossings = [], []
for d in D_LIST:
    for t in T_WINDOW:
        r = measure(d, t, store)
        grid.append(r)
        if r["exact"] and t > u_max:
            crossings.append(r)
        flag = "   <<< PAST THE BOUND" if r["exact"] and t > u_max else ""
        print(f"d={d}  t={t:>3}  ell={r['ell']:>3}  ker={r['kernel']:>4}  "
              f"roots={r['roots']:>3}  found={r['found']:>3}  "
              f"exact={str(r['exact']):>5}  {r['sec']:>7.1f}s{flag}", flush=True)
    print()

print("-" * 72)
if crossings:
    best = max(c["t"] for c in crossings)
    ds = sorted({c["d"] for c in crossings})
    print(f"the bound is u_max = {u_max}; exact recovery reached t = {best} "
          f"at d = {ds}")
else:
    print(f"no d in {D_LIST} recovered exactly past u_max = {u_max}")

In [ ]:
MODE = "light"
PALETTE = {
    "light": {"surface": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e",
              "grid": "#dcdbd6", "ok": "#2a78d6", "no": "#e8e7e2"},
    "dark":  {"surface": "#1a1a19", "ink": "#ffffff", "ink2": "#c3c2b7",
              "grid": "#3a3a38", "ok": "#3987e5", "no": "#2b2b29"},
}
PAL = PALETTE[MODE]
mpl.rcParams.update({
    "figure.facecolor": PAL["surface"], "axes.facecolor": PAL["surface"],
    "savefig.facecolor": PAL["surface"],
    "text.color": PAL["ink"], "axes.labelcolor": PAL["ink2"],
    "xtick.color": PAL["ink2"], "ytick.color": PAL["ink2"],
    "font.size": 10, "figure.dpi": 130,
})

ts = sorted({r["t"] for r in grid})
fig, ax = plt.subplots(figsize=(2.2 + 0.9 * len(ts), 1.0 + 0.55 * len(D_LIST)))
for yi, d in enumerate(D_LIST):
    for xi, t in enumerate(ts):
        r = next((g for g in grid if g["d"] == d and g["t"] == t), None)
        if r is None:
            continue
        ax.add_patch(plt.Rectangle((xi - .45, yi - .42), .9, .84,
                                   facecolor=PAL["ok"] if r["exact"] else PAL["no"],
                                   edgecolor=PAL["surface"], linewidth=1.6))
        ax.text(xi, yi, str(r["found"]), ha="center", va="center", fontsize=9,
                color=PAL["surface"] if r["exact"] else PAL["ink2"])

ax.axvline(ts.index(u_max) + 0.5, color=PAL["ink"], linestyle=(0, (4, 3)), linewidth=1.5)
ax.text(ts.index(u_max) + 0.62, len(D_LIST) - 0.35,
        f"theoretical bound\n$u_{{\\max}} = {u_max}$", color=PAL["ink"], fontsize=9,
        va="top", ha="left")
ax.set_xlim(-.6, len(ts) - .3)
ax.set_ylim(-.65, len(D_LIST) - .15)
ax.set_xticks(range(len(ts)), [str(t) for t in ts])
ax.set_yticks(range(len(D_LIST)), [f"d = {d}" for d in D_LIST])
ax.set_xlabel("t, codewords bundled")
ax.set_title(f"exact recovery at p,m,K = {P},{M},{K}"
             f"   (cell label = codewords returned)",
             color=PAL["ink"], loc="left", pad=12)
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(length=0)
fig.tight_layout()
(REPO / "figures").mkdir(exist_ok=True)
fig.savefig(REPO / "figures" / f"gs_bound_{P}_{M}_{K}.png", bbox_inches="tight")
plt.show()

## Reading the result

The dashed line is $u_{\max}$. A blue cell to its left is theory and practice agreeing.
A blue cell to its **right** is exact recovery at a $t$ the theory does not cover.

If you see one, nothing has been contradicted. §1 proves sufficiency only, and the two
arguments fail in different ways once $t > u_{\max}$:

- **(b) fails silently.** $Q^{*}$ no longer fits the budget, so the standard existence
  proof goes away --- but the counting cell shows the unknown count can still exceed the
  constraint count at larger $d$, and then *some* curve exists. It will not be $Q^{*}$.
- **(a) loses its guarantee, which is not the same as being wrong.** Once
  $t(K-1) > N$ an impostor *could* in principle reach the maximum score $dN$. Whether one
  actually does is a property of the particular codewords bundled, and near the bound
  there usually is none. That gap --- between "no longer provably safe" and "actually
  produces a wrong answer" --- is the entire source of any extra reach.

So the certification still holds for each individual answer: every codeword reported was
checked to score $dN$ against the observed histogram. What is lost past $u_{\max}$ is the
*a priori* promise, not the *a posteriori* check.

If you see no crossing, that is the other honest outcome: at this $(p, m, K)$ the
counting never gets far enough ahead, and $u_{\max}$ is tight in practice as well as in
theory.

Things to try: extend `D_LIST` (watch the $d^4$ cost), widen `T_WINDOW`, or change
$(P, M, K)$ at the top. $K = 2$ makes $u_{\max} = N - 1$ and the bundles enormous; larger
$K$ shrinks it and makes the whole experiment cheap enough to sweep properly.